# Perfil del snapshot raw OAI

Notebook de sólo lectura para controlar cobertura, procedencia y calidad antes de ejecutar `oai_load`. Ejecutar desde `kedro jupyter lab` después de una cosecha.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

In [ ]:
identifiers = catalog.load("raw/oai/identifiers#parquet")
records = catalog.load("raw/oai/records#parquet")
optional_datasets = {
    "missing_after_bulk": "raw/oai/missing_record_identifiers#parquet",
    "recovered": "raw/oai/records_recovered#parquet",
    "record_errors": "raw/oai/record_errors#parquet",
    "page_errors": "raw/oai/record_page_errors#parquet",
}
profile_data = {name: catalog.load(dataset) if catalog.exists(dataset) else pd.DataFrame() for name, dataset in optional_datasets.items()}
identifiers.head()

## Cobertura y reconciliación

In [ ]:
active_ids = identifiers.loc[~identifiers["is_deleted"].fillna(False), "record_id"]
remaining_ids = active_ids.loc[~active_ids.isin(records["record_id"])]
summary = pd.Series({
    "identifiers": len(identifiers),
    "active_identifiers": len(active_ids),
    "deleted_identifiers": int(identifiers["is_deleted"].fillna(False).sum()),
    "records": len(records),
    "unique_records": records["record_id"].nunique(),
    "duplicate_records": int(records["record_id"].duplicated().sum()),
    "missing_after_bulk": len(profile_data["missing_after_bulk"]),
    "recovered_by_get_record": len(profile_data["recovered"]),
    "remaining_active": len(remaining_ids),
    "record_errors": len(profile_data["record_errors"]),
    "page_errors": len(profile_data["page_errors"]),
    "active_coverage_pct": round(len(records) / len(active_ids) * 100, 4) if len(active_ids) else None,
}, name="value")
summary.to_frame()

In [ ]:
assert identifiers["record_id"].notna().all()
assert records["record_id"].notna().all()
assert not records["record_id"].duplicated().any()
assert set(remaining_ids) == set(profile_data["record_errors"].get("record_id", pd.Series(dtype=object)))
remaining_ids.to_frame(name="record_id")

## Procedencia y lote

In [ ]:
provenance_columns = ["_source_key", "_repository_identifier", "_institution_ror", "_base_url", "_metadata_prefix", "_context"]
provenance = pd.DataFrame({column: records[column].value_counts(dropna=False) for column in provenance_columns})
batch = records["_extract_datetime"].agg(["min", "max", "count"]).to_frame(name="value")
display(provenance)
display(batch)

## Completitud de Dublin Core

In [ ]:
metadata_columns = ["title", "date_issued", "creators", "description", "types", "identifiers", "languages", "subjects", "publishers", "relations", "rights", "formats", "set_id"]
completeness = {}
for column in metadata_columns:
    populated = records[column].map(lambda value: len(value) > 0 if isinstance(value, (list, tuple, np.ndarray)) else pd.notna(value) and str(value).strip() != "")
    completeness[column] = {"populated": int(populated.sum()), "missing": int((~populated).sum()), "coverage_pct": round(populated.mean() * 100, 2)}
pd.DataFrame(completeness).T.sort_values("coverage_pct")

## Distribuciones principales

In [ ]:
years = records["date_issued"].astype("string").str.extract(r"((?:19|20)\d{2})", expand=False)
display(years.value_counts(dropna=False).sort_index().rename_axis("year").to_frame("records"))
for column in ["set_id", "types", "languages", "rights", "formats", "subjects"]:
    top_values = records[["record_id", column]].explode(column).dropna(subset=[column])[column].value_counts().head(30).to_frame("records")
    display(column, top_values)

## Registros para inspección

In [ ]:
core_missing = records[records["title"].isna() | records["date_issued"].isna() | records["creators"].map(lambda value: len(value) == 0 if isinstance(value, (list, tuple, np.ndarray)) else pd.isna(value))]
core_missing[["record_id", "title", "date_issued", "creators", "types", "rights"]].head(50)

In [ ]:
display(profile_data["page_errors"])
display(profile_data["record_errors"])